In [ ]:
# Current conda environment does not support tmap, set this up separately
# !conda create -n tmap python=3.9
# !pip install ipykernel pandas faerun mhfp tqdm rdkit
# !conda install -c tmap tmap -y

In [4]:
from tqdm import tqdm
import numpy as np
import tmap as tm
from rdkit.Chem import AllChem
from mhfp.encoder import MHFPEncoder
from faerun import Faerun
from matplotlib.colors import ListedColormap

In [ ]:
# Cache fp results
fp_cache = dict()

# Parallel processing
# from joblib import Parallel, delayed
# import pandarallel

# pandarallel.pandarallel.initialize(progress_bar=True)

filter_types = [
    # "AChE (generated)",
    # "MAOB (generated)",
    # "AChE_MAOB_SUM (generated)",

    "D2R (generated)",
    "_5HT2A (generated)",
    "D2R__5HT2A_SUM (generated)",

    # "D2R (generated)",
    # "D3R (generated)",
    # "D2R_D3R_SUM (generated)",
]

colors = {
    # "AChE": "lightgreen",
    # "MAOB": "#ff9a98", # "lightred",
    "AChE (generated)": "#7FFFD4",
    "MAOB (generated)": "#9932CC",
    "AChE_MAOB_SUM (generated)": "#FF8C00",

    # "D2R": "orange",
    # "_5HT2A": "blue",
    "D2R (generated)": "#FFFF00",
    "_5HT2A (generated)": "#008080",
    "D2R__5HT2A_SUM (generated)": "#FF2400",

    # "D3R": "blue",
    # "D2R (generated)": "#00008B",
    # "D3R (generated)": "#FFD700",
    # "D2R_D3R_SUM (generated)": "#FF69B4",
}

df = training_mols_mt.query("target in @filter_types").reset_index(drop=True)
enc = MHFPEncoder(1024)
lf = tm.LSHForest(
    d=1024,  # d = dimensionality of the MinHashe vectors to be added
    l=64     # l = number of prefix trees used when indexing data
)

fps = list()

def mol_to_fp(smiles):
    if smiles in fp_cache:
        return fp_cache[smiles]

    try:
        mol = AllChem.MolFromSmiles(smiles)
        fp = tm.VectorUint(enc.encode_mol(mol))
        fp_cache[smiles] = fp
        return fp
    except:
        fp_cache[smiles] = None
        return None

# fps = Parallel(n_jobs=-1, backend="threading")(delayed(mol_to_fp)(smiles) for smiles in tqdm(df["SMILES"]))
# fps = [fp for fp in fps if fp is not None]

for smiles in tqdm(df["SMILES"]):
    fp = mol_to_fp(smiles)
    if fp is not None:
        fps.append(fp)

lf.batch_add(fps)
lf.index()

# Reference: https://github.com/reymond-group/tmap/blob/c74b718a86843292ab6aad91b99196b0133faac9/src/_tmap/layout.hh#L181
cfg = tm.LayoutConfiguration()

# The size of the nodes, which affects the magnitude of their repelling force.
# Decreasing this value generally resolves overlaps in a very crowded tree.
cfg.node_size = 1 / 26
# Number of repeats of the per-level layout algorithm
cfg.mmm_repeats = 2
# Sets the number of repeats of the scaling.
cfg.sl_extra_scaling_steps = 5
# The number of nearest neighbors used to create the k-nearest neighbor graph
cfg.k = 20
# Defines the (relative) scale of the graph
cfg.sl_scaling_type = tm.RelativeToAvgLength
# Returns: The x and y coordinates of the vertices, the ids of the vertices spanning the edges, and information on the graph
x, y, s, t, graph_properties = tm.layout_from_lsh_forest(lf, cfg)

type_labels, type_data = Faerun.create_categories(df["target"])

cmap = ListedColormap([colors[t[1]] for t in type_labels])

f = Faerun(view="front", coords=False, clear_color='#ffffff')
f.add_scatter(
    "np_atlas",
    {
        "x": x,
        "y": y,
        "c": [
            type_data,
        ],
        "labels": df["SMILES"],
    },
    shader="smoothCircle",
    point_scale=5.0,
    max_point_size=20,
    legend_labels=[type_labels],
    categorical=[True],
    colormap=[cmap],
    series_title=["Type",],
    has_legend=True,
)
f.add_tree("np_atlas_tree", {"from": s, "to": t}, point_helper="np_atlas")
f.plot(template="smiles", notebook_height=0)

In [ ]:
type_labels, type_data = Faerun.create_categories(training_mols["target"])
type_labels, len(graph_properties.adjacency_list), len(df)

In [ ]:
import networkx as nx
from collections import defaultdict
import pandas as pd

# Assume G is your graph and class_dict maps each node to its class
G = nx.Graph()
class_dict = {}

for i, row in df.iterrows():
    G.add_node(i)
    class_dict[i] = row['target']

for i, adj in enumerate(graph_properties.adjacency_list):
    # adj = adj[0]
    for j in adj:
        G.add_edge(i, j[0], weight=j[1])
        # G.add_edge(i, j[0])

node_classes = class_dict
nx.set_node_attributes(G, node_classes, 'class')
print(len(G.nodes), len(G.edges))